In [ ]:
# Generate 2D random matrices from Gaussian random fields, then split them into 1D samples (sample groups)
'''
modified from code of Bruno Sciolla, https://github.com/bsciolla/gaussian-random-fields
'''

# Main dependencies
import numpy as np
import scipy.fftpack
import matplotlib
import matplotlib.pyplot as plt
import scipy.io as sio
import pandas as pd


def fftind(size):
    """ Returns a numpy array of shifted Fourier coordinates k_x k_y.
        
        Input args:
            size (integer): The size of the coordinate array to create
        Returns:
            k_ind, numpy array of shape (2, size, size) with:
                k_ind[0,:,:]:  k_x components
                k_ind[1,:,:]:  k_y components
                
        Example:
        
            print(fftind(5))
            
            [[[ 0  1 -3 -2 -1]
            [ 0  1 -3 -2 -1]
            [ 0  1 -3 -2 -1]
            [ 0  1 -3 -2 -1]
            [ 0  1 -3 -2 -1]]

            [[ 0  0  0  0  0]
            [ 1  1  1  1  1]
            [-3 -3 -3 -3 -3]
            [-2 -2 -2 -2 -2]
            [-1 -1 -1 -1 -1]]]
            
        """
    k_ind = np.mgrid[:size, :size] - int( (size + 1)/2 )
    k_ind = scipy.fftpack.fftshift(k_ind)
    return( k_ind )



# return numpy.ndarray of shape (size, size),
def gaussian_random_field(alpha = 6.5,# smooth factor
                          size = 64, # size of the field
                          mode = 'bound',# 'random' or bound
                          set_1 = 1.0, # if 'random', mean; else if 'bound', lower bound
                          set_2 = 1000.0):# if 'random', standard derivation; else if 'bound', upper bound
    """ Returns a numpy array of shifted Fourier coordinates k_x k_y.
        
        Input args:
            alpha (double, default = 3.0): 
                The power of the power-law momentum distribution
            size (integer, default = 128):
                The size of the square output Gaussian Random Fields
            flag_normalize (boolean, default = True):
                Normalizes the Gaussian Field:
                    - to have an average of 0.0
                    - to have a standard deviation of 1.0

        Returns:
            gfield (numpy array of shape (size, size)):
                The random gaussian random field
                
        Example:
        import matplotlib
        import matplotlib.pyplot as plt
        example = gaussian_random_field()
        plt.imshow(example)
        """
        
        # Defines momentum indices
    k_idx = fftind(size)

        # Defines the amplitude as a power law 1/|k|^(alpha/2)
    amplitude = np.power( k_idx[0]**2 + k_idx[1]**2 + 1e-10, -alpha/4.0 )
    amplitude[0,0] = 0
    
        # Draws a complex gaussian random noise with normal
        # (circular) distribution
    noise = np.random.normal(size = (size, size)) \
        + 1j * np.random.normal(size = (size, size))
    
        # To real space
    gfield = np.fft.ifft2(noise * amplitude).real
    
        # Sets the standard deviation to one
    gfield = gfield - np.mean(gfield)
    gfield = gfield/np.std(gfield)
    if mode == 'random':
        set_mean = set_1
        set_std  = set_2
        gfield = gfield * set_std
        gfield = gfield + set_mean
    elif mode == 'bound':
        set_lower = set_1
        set_upper = set_2
        g_max = np.max(gfield)
        g_min = np.min(gfield)
        gfield = (set_upper-set_lower)/(g_max - g_min) * gfield + \
                (set_lower*g_max - set_upper*g_min)/(g_max - g_min)
    else:
          raise KeyError("mode must be 'random' or 'bound")   
    return gfield




alphas = [1, 3, 5, 7, 9]
samples_per_alpha = 2
size = 64

# Two rows and five columns: each column corresponds to one alpha; the first row is sample 1 and the second row is sample 2

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "mathtext.fontset": "stix",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "axes.unicode_minus": False,
})

fig, axes = plt.subplots(2, len(alphas), figsize=(19, 8))
# fig.suptitle('Gaussian Random Fields for Different Alpha Values', fontsize=18, y=0.96)

im_last = None  # Store the image handle from the last column for a shared colorbar

for col, alpha in enumerate(alphas):
    for row in range(samples_per_alpha):
        grf = gaussian_random_field(
            alpha=alpha, size=size, mode='bound', set_1=1.0, set_2=1000.0
        )

        ax = axes[row, col]
        im = ax.imshow(grf, cmap='jet')
        # ax.set_title(f'alpha={alpha}', fontsize=11)
        ax.set_title(f'$\\alpha={alpha}$', fontsize=14)
        ax.set_xticks([])
        ax.set_yticks([])

        if col == len(alphas) - 1:
            im_last = im

# Reduce subplot spacing and reserve space on the right for the colorbar
fig.subplots_adjust(left=0.03, right=0.92, bottom=0.05, top=0.90, wspace=0.05, hspace=0.08)

# Place one independent colorbar on the far right without squeezing the last column of subplots
cax = fig.add_axes([0.935, 0.12, 0.012, 0.72])  # [left, bottom, width, height]
cbar = fig.colorbar(im_last, cax=cax)
cbar.ax.tick_params(labelsize=8)

# plt.savefig('alpha_compare_2x5_lastcol_colorbar.png', dpi=300, bbox_inches='tight')
# Save as a PNG bitmap
plt.savefig('F1.png', dpi=600, bbox_inches='tight')

# Save as a vector graphic named F1
plt.savefig('F1.svg', bbox_inches='tight')

plt.show()



In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ===== Load data =====
S = np.load("S.npy")
grf_array = np.load("grf_array.npy")
corr = np.load("corr.npy", mmap_mode="r")

# ===== Specify the ranks to display (1-based, fixed at 3) =====
selected_ranks = [3, 1, 4]
# ============================================

S_arr = np.asarray(S)
if S_arr.size == 0:
    raise ValueError("S 为空：没有可展示的配对。")
if S_arr.ndim == 1:
    S_arr = S_arr.reshape(-1, 2)
if len(selected_ranks) != 3:
    raise ValueError("selected_ranks 必须是 3 个元素，对应两行三列。")

idx_i = S_arr[:, 0].astype(int)
idx_j = S_arr[:, 1].astype(int)
r_vals = corr[idx_i, idx_j]

# Sort by |r| in descending order
order = np.argsort(np.abs(r_vals))[::-1]
S_sorted = S_arr[order]
r_sorted = r_vals[order]

# User-specified ranks (1-based -> 0-based)
sel = np.asarray(selected_ranks, dtype=int) - 1
if np.any(sel < 0) or np.any(sel >= len(S_sorted)):
    raise IndexError(f"selected_ranks 越界，当前可选范围是 1 ~ {len(S_sorted)}")

pairs_to_show = S_sorted[sel]
r_show = r_sorted[sel]

# ===== Plot in two rows and three columns =====

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "mathtext.fontset": "stix",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "axes.unicode_minus": False,
})


fig, axes = plt.subplots(2, 3, figsize=(12, 8))

im_last = None

for col in range(3):
    i, j = map(int, pairs_to_show[col])
    r = float(r_show[col])
    p = r

    im = axes[0, col].imshow(
        grf_array[:, :, i],
        cmap="jet",
        vmin=1.0,
        vmax=1000.0
    )

    axes[1, col].imshow(
        grf_array[:, :, j],
        cmap="jet",
        vmin=1.0,
        vmax=1000.0
    )

    # ===== Updated titles =====
    # axes[0, col].set_title(f"X[{i}]  p={p:.4f}", fontsize=12)
    # axes[1, col].set_title(f"Y[{j}]  p={p:.4f}", fontsize=12)
    axes[0, col].set_title(fr"$\mathrm{{X}}[{i}]$  $p={p:.4f}$", fontsize=14)
    axes[1, col].set_title(fr"$\mathrm{{Y}}[{j}]$  $p={p:.4f}$", fontsize=14)

    axes[0, col].set_xticks([])
    axes[0, col].set_yticks([])
    axes[1, col].set_xticks([])
    axes[1, col].set_yticks([])

    if col == 2:
        im_last = im

# Reduce subplot spacing and place an independent colorbar on the far right
fig.subplots_adjust(
    left=0.03,
    right=0.88,
    bottom=0.05,
    top=0.93,
    wspace=0.05,
    hspace=0.06
)

cax = fig.add_axes([0.895, 0.12, 0.015, 0.72])
cbar = fig.colorbar(im_last, cax=cax)
cbar.ax.tick_params(labelsize=8)

# Save figures
plt.savefig("F2.png", dpi=600, bbox_inches="tight")
plt.savefig("F2.svg", bbox_inches="tight")

plt.show()